# CNNのパラメータ数と学習の仕組み

このノートブックでは、CNNに必要なパラメータの数（図3.18）と、誤差を最小化して画像分類の精度を向上させる仕組み（3-7節、3-8節）を学びます。

## 目次
1. CNNアーキテクチャとパラメータ数（図3.18）
2. AIのブラックボックス化
3. 誤差を最小化して精度を向上させる（3-7節）
4. 損失関数と勾配降下法
5. 簡略化したCNNモデル（図3.20）
6. 損失関数を定義する―対数尤度関数―（3-8節）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, FancyArrowPatch
import matplotlib.patches as mpatches

plt.rcParams['font.size'] = 10

## 1. CNNアーキテクチャとパラメータ数（図3.18）

手書き数字「8」を認識するCNNの構造と、各層のパラメータ数を確認します。

### CNNの全体構造

| 層 | 入力サイズ | 処理 | 出力サイズ | 重みパラメータ | バイアス |
|---|---|---|---|---|---|
| 入力層 | - | - | 24×24 | - | - |
| 畳み込み層1 | 24×24 | 5×5フィルタ×2, stride=1 | 20×20×2 | 50 | 2 |
| プーリング層1 | 20×20×2 | 2×2, stride=2 | 10×10×2 | 0 | 0 |
| 畳み込み層2 | 10×10×2 | 3×3フィルタ×8, stride=1 | 8×8×4 | 72 | 4 |
| プーリング層2 | 8×8×4 | 2×2, stride=2 | 4×4×4 | 0 | 0 |
| 全結合層 | 64 | 64→10 | 10 | 640 | 10 |
| 出力層 | 10 | ソフトマックス | 10 | - | - |

In [ ]:
# 図3.18: CNNアーキテクチャの可視化

fig, ax = plt.subplots(figsize=(18, 10))

# 各層の情報
layers = [
    {'name': '入力層\n24×24', 'x': 0, 'size': (2.4, 2.4), 'color': 'lightgray', 'weights': '-', 'bias': '-'},
    {'name': '畳み込み層\n5×5×2\nstride=1', 'x': 3, 'size': (2.0, 2.0), 'color': 'steelblue', 'weights': '50', 'bias': '2'},
    {'name': '20×20×2', 'x': 3, 'size': (2.0, 2.0), 'color': 'steelblue', 'is_output': True},
    {'name': 'プーリング層\n2×2\nstride=2', 'x': 6, 'size': (1.0, 1.0), 'color': 'coral', 'weights': '0', 'bias': '0'},
    {'name': '10×10×2', 'x': 6, 'size': (1.0, 1.0), 'color': 'coral', 'is_output': True},
    {'name': '畳み込み層\n3×3×8\nstride=1', 'x': 9, 'size': (0.8, 0.8), 'color': 'steelblue', 'weights': '72', 'bias': '4'},
    {'name': '8×8×4', 'x': 9, 'size': (0.8, 0.8), 'color': 'steelblue', 'is_output': True},
    {'name': 'プーリング層\n2×2\nstride=2', 'x': 12, 'size': (0.4, 0.4), 'color': 'coral', 'weights': '0', 'bias': '0'},
    {'name': '4×4×4', 'x': 12, 'size': (0.4, 0.4), 'color': 'coral', 'is_output': True},
    {'name': '全結合層\n64→10', 'x': 15, 'size': (0.3, 2.0), 'color': 'purple', 'weights': '640', 'bias': '10'},
    {'name': '出力層\nSoftmax', 'x': 17, 'size': (0.3, 2.0), 'color': 'green', 'weights': '-', 'bias': '-'},
]

center_y = 5
drawn_layers = []

# 層を描画（is_output=Trueの層はスキップ）
for layer in layers:
    if layer.get('is_output'):
        continue
    
    x = layer['x']
    w, h = layer['size']
    
    # 層のボックスを描画
    rect = FancyBboxPatch((x - w/2, center_y - h/2), w, h,
                          boxstyle="round,pad=0.02",
                          facecolor=layer['color'], edgecolor='black',
                          linewidth=2, alpha=0.8)
    ax.add_patch(rect)
    
    # 層名を下に
    ax.text(x, center_y - h/2 - 0.3, layer['name'], 
            ha='center', va='top', fontsize=9, fontweight='bold')
    
    drawn_layers.append(layer)

# 矢印を追加
arrow_pairs = [(0, 3), (3, 6), (6, 9), (9, 12), (12, 15), (15, 17)]
for x1, x2 in arrow_pairs:
    # 対応する層を見つける
    l1 = next((l for l in layers if l['x'] == x1 and not l.get('is_output')), None)
    l2 = next((l for l in layers if l['x'] == x2 and not l.get('is_output')), None)
    if l1 and l2:
        w1 = l1['size'][0]
        w2 = l2['size'][0]
        ax.annotate('', xy=(x2 - w2/2 - 0.1, center_y),
                    xytext=(x1 + w1/2 + 0.1, center_y),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# パラメータ情報を上部に追加
param_info = [
    (3, '重み: 50個\nバイアス: 2個', 'steelblue'),
    (9, '重み: 72個\nバイアス: 4個', 'steelblue'),
    (15, '重み: 640個\nバイアス: 10個', 'purple'),
]

for x, text, color in param_info:
    ax.text(x, center_y + 2.5, text, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, linewidth=2))

# 「8」の入力画像を描画
from matplotlib.patches import Circle

# 8の形を描画（入力層の位置に）
img_center_x = 0
circle_top = Circle((img_center_x, center_y + 0.5), 0.4, fill=False, color='black', linewidth=3)
circle_bottom = Circle((img_center_x, center_y - 0.5), 0.5, fill=False, color='black', linewidth=3)
ax.add_patch(circle_top)
ax.add_patch(circle_bottom)

# 出力確率を右側に表示
probs = [0.05, 0.05, 0.10, 0.05, 0.10, 0.05, 0.05, 0.05, 0.43, 0.05]  # P8が最大
output_x = 18.5
for i, p in enumerate(probs):
    color = 'red' if i == 8 else 'black'
    weight = 'bold' if i == 8 else 'normal'
    ax.text(output_x, center_y + 2.5 - i * 0.5, f'P{i} = {p:.2f}', 
            fontsize=10, color=color, fontweight=weight)

# タイトルと凡例
ax.set_title('図3.18: CNNに必要となるパラメータ数の例', fontsize=14, fontweight='bold', pad=20)

# 合計パラメータ数を下部に表示
ax.text(9, center_y - 4, 
        '重みパラメータ合計: 50 + 72 + 640 = 762個\n'
        'バイアス合計: 2 + 4 + 10 = 16個\n'
        '総パラメータ数: 778個',
        ha='center', va='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', linewidth=2))

ax.set_xlim(-2, 21)
ax.set_ylim(-1, 9)
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.show()

### パラメータ数の計算方法

各層のパラメータ数がどのように計算されるか、詳しく見てみましょう。

In [ ]:
# 各層のパラメータ数を詳しく計算

print("=" * 60)
print("CNNパラメータ数の詳細計算（図3.18）")
print("=" * 60)

print("\n【畳み込み層1】")
print("  入力: 24×24×1（グレースケール画像）")
print("  フィルタ: 5×5, 2枚")
print("  重み = フィルタサイズ × 入力チャンネル × 出力チャンネル")
print("       = 5 × 5 × 1 × 2 = 50個")
print("  バイアス = 出力チャンネル数 = 2個")
conv1_weights = 5 * 5 * 1 * 2
conv1_bias = 2

print("\n【プーリング層1】")
print("  2×2 Max Pooling (stride=2)")
print("  パラメータ: 0個（学習するパラメータなし）")

print("\n【畳み込み層2】")
print("  入力: 10×10×2")
print("  フィルタ: 3×3, 4枚（2チャンネル入力から4チャンネル出力）")
print("  重み = 3 × 3 × 2 × 4 = 72個")
print("  バイアス = 4個")
conv2_weights = 3 * 3 * 2 * 4
conv2_bias = 4

print("\n【プーリング層2】")
print("  2×2 Max Pooling (stride=2)")
print("  パラメータ: 0個")

print("\n【全結合層】")
print("  入力: 4×4×4 = 64ニューロン（平坦化後）")
print("  出力: 10クラス（0〜9の数字）")
print("  重み = 入力数 × 出力数 = 64 × 10 = 640個")
print("  バイアス = 10個")
fc_weights = 64 * 10
fc_bias = 10

print("\n" + "=" * 60)
print("【合計】")
total_weights = conv1_weights + conv2_weights + fc_weights
total_bias = conv1_bias + conv2_bias + fc_bias
total_params = total_weights + total_bias
print(f"  重みパラメータ: {conv1_weights} + {conv2_weights} + {fc_weights} = {total_weights}個")
print(f"  バイアス: {conv1_bias} + {conv2_bias} + {fc_bias} = {total_bias}個")
print(f"  ────────────────────────────────────────")
print(f"  総パラメータ数: {total_params}個")
print("=" * 60)

In [ ]:
# パラメータ数の内訳を可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 重みパラメータの内訳
ax = axes[0]
layers_names = ['畳み込み層1\n(5×5×2)', '畳み込み層2\n(3×3×8)', '全結合層\n(64→10)']
weights = [50, 72, 640]
colors = ['steelblue', 'steelblue', 'purple']
bars = ax.bar(layers_names, weights, color=colors, edgecolor='black', linewidth=2)

# 値をバーの上に表示
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
            f'{w}個', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('パラメータ数', fontsize=12)
ax.set_title('重みパラメータの内訳\n合計: 762個', fontsize=12, fontweight='bold')
ax.set_ylim(0, 750)
ax.grid(axis='y', alpha=0.3)

# 右: 円グラフ
ax = axes[1]
sizes = [50, 72, 640]
labels = ['畳み込み層1\n(50個, 6.6%)', '畳み込み層2\n(72個, 9.4%)', '全結合層\n(640個, 84.0%)']
colors = ['steelblue', 'deepskyblue', 'purple']
explode = (0, 0, 0.1)  # 全結合層を強調

wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors,
                                   autopct='', startangle=90, 
                                   wedgeprops=dict(edgecolor='black', linewidth=2))

ax.set_title('重みパラメータの割合\n→ 全結合層が84%を占める！', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n【重要な発見】")
print(f"全結合層のパラメータ数: {640}/{762} = {640/762*100:.1f}%")
print("→ パラメータのほとんどは全結合層にある！")
print("→ これが深層学習モデルが大きくなる主な原因の一つ")

## 2. AIのブラックボックス化

### なぜAIは「ブラックボックス」と呼ばれるのか？

上記の例では778個のパラメータを使用していますが、これは非常に小さな数です。実際のCNNモデルでは：

- **簡単なモデル**: 数万〜数十万パラメータ
- **標準的なモデル**: 数百万〜数千万パラメータ
- **大規模モデル**: 数億〜数兆パラメータ（GPT、LLMなど）

これらすべてのパラメータが学習によって更新され、「なぜその値なのか」を人間が解釈することは事実上不可能です。

In [ ]:
# 有名なモデルのパラメータ数比較

models = {
    'この教科書の例': 778,
    'LeNet-5 (1998)': 60_000,
    'AlexNet (2012)': 60_000_000,
    'VGG-16 (2014)': 138_000_000,
    'ResNet-152 (2015)': 60_000_000,
    'GPT-3 (2020)': 175_000_000_000,
}

fig, ax = plt.subplots(figsize=(12, 6))

names = list(models.keys())
params = list(models.values())

# 対数スケールでプロット
bars = ax.barh(names, params, color=['red'] + ['steelblue'] * 5)
ax.set_xscale('log')
ax.set_xlabel('パラメータ数（対数スケール）', fontsize=12)
ax.set_title('有名なディープラーニングモデルのパラメータ数比較', fontsize=14, fontweight='bold')

# 値をバーの右に表示
for bar, p in zip(bars, params):
    if p >= 1_000_000_000:
        label = f'{p/1_000_000_000:.0f}B'
    elif p >= 1_000_000:
        label = f'{p/1_000_000:.0f}M'
    elif p >= 1_000:
        label = f'{p/1_000:.0f}K'
    else:
        label = str(p)
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2, 
            label, va='center', fontsize=11)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n778個のパラメータでさえ、各値の意味を完全に理解することは困難です。")
print("何十億、何兆ものパラメータの解釈は、もはや人間の理解を超えています。")
print("\nこれが『AIのブラックボックス化』の本質です。")

## 3. 誤差を最小化して精度を向上させる（3-7節）

CNNなどの深層学習モデルは、**誤差を最小化**することで学習します。

### 学習の基本的な考え方

1. **予測値**: モデルが出力した確率（例: P(8) = 0.43）
2. **正解値**: 実際のラベル（例: 正解は「8」なので理想的にはP(8) = 1.0）
3. **誤差（損失）**: 予測値と正解値の差を数値化したもの
4. **学習**: 誤差が小さくなるようにパラメータを更新

In [ ]:
# 予測と正解の比較

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

classes = list(range(10))

# 左: 現在の予測
ax = axes[0]
pred = [0.05, 0.05, 0.10, 0.05, 0.10, 0.05, 0.05, 0.05, 0.43, 0.07]
colors = ['coral' if i == 8 else 'steelblue' for i in range(10)]
ax.bar(classes, pred, color=colors, edgecolor='black')
ax.set_xlabel('クラス（数字）')
ax.set_ylabel('確率')
ax.set_title('モデルの予測\nP(8) = 0.43', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_xticks(classes)

# 中央: 理想的な正解
ax = axes[1]
target = [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]  # One-hot encoding
colors = ['coral' if i == 8 else 'gray' for i in range(10)]
ax.bar(classes, target, color=colors, edgecolor='black')
ax.set_xlabel('クラス（数字）')
ax.set_ylabel('確率')
ax.set_title('理想的な出力（正解）\nP(8) = 1.00', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_xticks(classes)

# 右: 誤差
ax = axes[2]
error = [t - p for t, p in zip(target, pred)]
colors = ['red' if e > 0 else 'blue' for e in error]
ax.bar(classes, error, color=colors, edgecolor='black')
ax.set_xlabel('クラス（数字）')
ax.set_ylabel('誤差 (正解 - 予測)')
ax.set_title('誤差\nこれを小さくしたい！', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylim(-0.2, 0.7)
ax.set_xticks(classes)

plt.tight_layout()
plt.show()

print("赤: P(8)をもっと上げたい (+0.57)")
print("青: 他のクラスの確率を下げたい")

## 4. 損失関数と勾配降下法

### 損失関数（Loss Function）

誤差を数値化する関数です。分類問題では**交差エントロピー損失**がよく使われます。

$$L = -\sum_{i} y_i \log(p_i)$$

- $y_i$: 正解ラベル（one-hot encoding）
- $p_i$: 予測確率

正解クラスの予測確率が高いほど、損失は小さくなります。

In [ ]:
# 交差エントロピー損失の計算

def cross_entropy_loss(y_true, y_pred):
    """交差エントロピー損失を計算
    
    Args:
        y_true: 正解ラベル（one-hot）
        y_pred: 予測確率
    Returns:
        損失値
    """
    # ゼロ除算を防ぐため、小さな値を加える
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred))

# 例: 正解が「8」の場合
y_true = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0])  # 正解は8

print("=== 交差エントロピー損失の例 ===")
print(f"正解: 8")
print()

# 異なる予測での損失
predictions = [
    ([0.05, 0.05, 0.10, 0.05, 0.10, 0.05, 0.05, 0.05, 0.43, 0.07], "現在の予測 P(8)=0.43"),
    ([0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.80, 0.06], "改善後 P(8)=0.80"),
    ([0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.95, 0.01], "理想に近い P(8)=0.95"),
    ([0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10], "全て均等 P(8)=0.10"),
]

for pred, desc in predictions:
    loss = cross_entropy_loss(y_true, np.array(pred))
    print(f"{desc}")
    print(f"  損失 = -log({pred[8]:.2f}) = {loss:.4f}")
    print()

In [ ]:
# 損失関数の可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: -log(p) のグラフ
ax = axes[0]
p = np.linspace(0.01, 1, 100)
loss = -np.log(p)
ax.plot(p, loss, 'b-', linewidth=2)
ax.axvline(x=0.43, color='red', linestyle='--', alpha=0.7, label='P(8)=0.43')
ax.axvline(x=0.95, color='green', linestyle='--', alpha=0.7, label='P(8)=0.95')
ax.scatter([0.43, 0.95], [-np.log(0.43), -np.log(0.95)], s=100, zorder=5)
ax.set_xlabel('予測確率 P(正解クラス)', fontsize=12)
ax.set_ylabel('損失 = -log(P)', fontsize=12)
ax.set_title('交差エントロピー損失\n確率が高いほど損失は小さい', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 5)

# 右: 学習による損失の減少
ax = axes[1]
epochs = np.arange(0, 100)
# 学習曲線をシミュレート
initial_loss = 2.3  # 均等分布の場合の損失
final_loss = 0.1
loss_curve = initial_loss * np.exp(-0.05 * epochs) + final_loss * (1 - np.exp(-0.05 * epochs))
# ノイズを追加
np.random.seed(42)
loss_curve += np.random.normal(0, 0.05, len(epochs))
loss_curve = np.maximum(loss_curve, 0.05)

ax.plot(epochs, loss_curve, 'b-', linewidth=2)
ax.fill_between(epochs, loss_curve, alpha=0.3)
ax.axhline(y=final_loss, color='green', linestyle='--', alpha=0.7, label='目標損失')
ax.set_xlabel('学習回数（エポック）', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('学習による損失の減少\nパラメータを更新して損失を下げる', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【ポイント】")
print("・損失が小さい = モデルの予測が正確")
print("・学習 = 損失を最小化するようにパラメータを更新すること")

### 勾配降下法（Gradient Descent）

損失を最小化するためにパラメータを更新する方法です。

$$w_{new} = w_{old} - \eta \cdot \frac{\partial L}{\partial w}$$

- $w$: 重みパラメータ
- $\eta$: 学習率（どれだけ更新するか）
- $\frac{\partial L}{\partial w}$: 勾配（損失が最も減る方向）

In [ ]:
# 勾配降下法の可視化（1次元の例）

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 損失関数（2次関数で近似）
def loss_func(w):
    return (w - 2) ** 2 + 0.5

def loss_gradient(w):
    return 2 * (w - 2)

# 左: 損失関数と勾配降下の軌跡
ax = axes[0]
w = np.linspace(-2, 6, 100)
L = loss_func(w)
ax.plot(w, L, 'b-', linewidth=2, label='損失関数 L(w)')

# 勾配降下法のシミュレーション
learning_rate = 0.3
w_history = [5.0]  # 初期値
for _ in range(10):
    grad = loss_gradient(w_history[-1])
    w_new = w_history[-1] - learning_rate * grad
    w_history.append(w_new)

# 軌跡をプロット
for i in range(len(w_history) - 1):
    w1, w2 = w_history[i], w_history[i+1]
    ax.annotate('', xy=(w2, loss_func(w2)), xytext=(w1, loss_func(w1)),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.scatter(w1, loss_func(w1), color='red', s=50, zorder=5)

ax.scatter(w_history[-1], loss_func(w_history[-1]), color='green', s=100, zorder=5, label='最適値')
ax.axvline(x=2, color='green', linestyle='--', alpha=0.5)
ax.set_xlabel('重みパラメータ w', fontsize=12)
ax.set_ylabel('損失 L(w)', fontsize=12)
ax.set_title('勾配降下法\n勾配の反対方向に進む', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 右: 学習率の影響
ax = axes[1]
ax.plot(w, L, 'b-', linewidth=2, label='損失関数')

learning_rates = [0.1, 0.3, 0.9]
colors = ['orange', 'red', 'purple']

for lr, color in zip(learning_rates, colors):
    w_hist = [5.0]
    for _ in range(5):
        grad = loss_gradient(w_hist[-1])
        w_new = w_hist[-1] - lr * grad
        w_hist.append(w_new)
    
    for i in range(len(w_hist) - 1):
        if i == 0:
            ax.plot([w_hist[i], w_hist[i+1]], [loss_func(w_hist[i]), loss_func(w_hist[i+1])],
                    color=color, linewidth=2, marker='o', markersize=5, label=f'η={lr}')
        else:
            ax.plot([w_hist[i], w_hist[i+1]], [loss_func(w_hist[i]), loss_func(w_hist[i+1])],
                    color=color, linewidth=2, marker='o', markersize=5)

ax.set_xlabel('重みパラメータ w', fontsize=12)
ax.set_ylabel('損失 L(w)', fontsize=12)
ax.set_title('学習率ηの影響\n大きすぎると不安定、小さすぎると遅い', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 6)

plt.tight_layout()
plt.show()

print("【学習率の選択】")
print("・η=0.1: 安定だが収束が遅い")
print("・η=0.3: バランスが良い")
print("・η=0.9: 収束が速いが不安定になる可能性")

### 778個のパラメータの学習

実際のCNNでは、778個すべてのパラメータに対して同時に勾配を計算し、更新します。

これを**誤差逆伝播法（Backpropagation）**と呼びます。

In [ ]:
# 学習プロセスのイメージ

fig, ax = plt.subplots(figsize=(16, 8))

# フローチャートを描画
boxes = [
    {'text': '入力画像\n「8」', 'x': 1, 'y': 5, 'color': 'lightgray'},
    {'text': 'CNN\n(778パラメータ)', 'x': 4, 'y': 5, 'color': 'steelblue'},
    {'text': '予測\nP(8)=0.43', 'x': 7, 'y': 5, 'color': 'coral'},
    {'text': '正解\nP(8)=1.00', 'x': 7, 'y': 2, 'color': 'lightgreen'},
    {'text': '損失計算\nL=0.84', 'x': 10, 'y': 3.5, 'color': 'gold'},
    {'text': '勾配計算\n∂L/∂w', 'x': 13, 'y': 3.5, 'color': 'plum'},
    {'text': 'パラメータ更新\nw = w - η∂L/∂w', 'x': 4, 'y': 1, 'color': 'lightblue'},
]

for box in boxes:
    rect = FancyBboxPatch((box['x'] - 1, box['y'] - 0.8), 2, 1.6,
                          boxstyle="round,pad=0.1",
                          facecolor=box['color'], edgecolor='black',
                          linewidth=2)
    ax.add_patch(rect)
    ax.text(box['x'], box['y'], box['text'], ha='center', va='center',
            fontsize=10, fontweight='bold')

# 矢印
arrows = [
    (2, 5, 3, 5),      # 入力 → CNN
    (5, 5, 6, 5),      # CNN → 予測
    (8, 5, 9, 4.2),    # 予測 → 損失
    (8, 2, 9, 2.8),    # 正解 → 損失
    (11, 3.5, 12, 3.5), # 損失 → 勾配
    (13, 2.7, 5.5, 1.5),  # 勾配 → 更新（曲線）
    (3, 1.5, 3.2, 4.2),  # 更新 → CNN（上向き）
]

for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# ラベル
ax.text(6.5, 6.5, '順伝播（Forward）', fontsize=12, fontweight='bold', color='blue')
ax.text(10, 0.5, '逆伝播（Backward）', fontsize=12, fontweight='bold', color='red')

# 円で囲む
from matplotlib.patches import FancyArrowPatch
ax.annotate('', xy=(3.5, 1.8), xytext=(14, 3.5),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3',
                            color='red', lw=2))

ax.set_xlim(-1, 16)
ax.set_ylim(-0.5, 7.5)
ax.set_title('CNNの学習プロセス（1回の学習ステップ）', fontsize=14, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.show()

print("【学習の流れ】")
print("1. 順伝播: 画像を入力し、予測を計算")
print("2. 損失計算: 予測と正解の差を計算")
print("3. 逆伝播: 各パラメータの勾配を計算")
print("4. 更新: 勾配に基づいてパラメータを更新")
print("5. これを繰り返して損失を最小化！")

## 5. 簡略化したCNNモデル（図3.20）

誤差逆伝播法を理解しやすくするために、簡略化したCNNを考えます。

### 簡略化CNNの構成

| 層 | 説明 |
|---|---|
| **入力層** | 3×3ピクセルの画像。0, 1, 2の3種類の数字を識別 |
| **畳み込み層** | 2×2フィルタを2枚使用、stride=1 |
| **プーリング層** | 2×2のMax Pooling |
| **全結合層** | 特徴量を1次元に配列 |
| **出力層** | ソフトマックス関数で3クラスの確率を出力 |

### 記号の定義

- $x_{i,j}$: 入力画像の$(i, j)$位置のピクセル値
- $w_{i,j}^k$: $k$枚目の畳み込みフィルタの$(i, j)$位置の重み
- $c_{i,j}^k$: $k$枚目のフィルタによる畳み込み出力（ReLU適用後）
- $z_1, z_2$: Max Pooling後の出力
- $w_{i,j}^{fc}$: 全結合層の重み（$i$は入力、$j$は出力のインデックス）
- $fc_0, fc_1, fc_2$: 全結合層の出力
- $P_0, P_1, P_2$: ソフトマックス関数による最終予測確率
- $t_0, t_1, t_2$: 正解ラベル（one-hot encoding）

In [ ]:
# 図3.20: 簡略化したCNNの可視化

fig, ax = plt.subplots(figsize=(18, 10))

# 入力層 (3x3)
input_x, input_y = 1, 5
cell_size = 0.4
for i in range(3):
    for j in range(3):
        rect = plt.Rectangle((input_x + j*cell_size, input_y - i*cell_size), 
                              cell_size, cell_size, fill=True, 
                              facecolor='lightblue' if (i+j) % 2 == 0 else 'white',
                              edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        ax.text(input_x + j*cell_size + cell_size/2, input_y - i*cell_size + cell_size/2,
                f'$x_{{{i+1},{j+1}}}$', ha='center', va='center', fontsize=8)

ax.text(input_x + 0.6, input_y + 0.8, '入力層\n3×3', ha='center', fontsize=10, fontweight='bold')

# 畳み込み層のフィルタ (2x2 x 2枚)
conv_x = 3.5
for filt in range(2):
    for i in range(2):
        for j in range(2):
            y_offset = 1.5 if filt == 0 else -1.5
            color = 'lightyellow' if filt == 0 else 'lightgreen'
            rect = plt.Rectangle((conv_x + j*cell_size, input_y + y_offset - i*cell_size), 
                                  cell_size, cell_size, fill=True, 
                                  facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            ax.text(conv_x + j*cell_size + cell_size/2, input_y + y_offset - i*cell_size + cell_size/2,
                    f'$w^{filt+1}_{{{i+1},{j+1}}}$', ha='center', va='center', fontsize=7)

ax.text(conv_x + 0.4, input_y + 2.8, 'フィルタ1', ha='center', fontsize=9)
ax.text(conv_x + 0.4, input_y - 1.0, 'フィルタ2', ha='center', fontsize=9)

# 畳み込み出力 (2x2 x 2枚)
conv_out_x = 6
for filt in range(2):
    for i in range(2):
        for j in range(2):
            y_offset = 1.5 if filt == 0 else -1.5
            color = 'lightyellow' if filt == 0 else 'lightgreen'
            rect = plt.Rectangle((conv_out_x + j*cell_size, input_y + y_offset - i*cell_size), 
                                  cell_size, cell_size, fill=True, 
                                  facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            ax.text(conv_out_x + j*cell_size + cell_size/2, input_y + y_offset - i*cell_size + cell_size/2,
                    f'$c^{filt+1}_{{{i+1},{j+1}}}$', ha='center', va='center', fontsize=7)

ax.text(conv_out_x + 0.4, input_y + 2.8, '畳み込み出力', ha='center', fontsize=9)
ax.text(conv_x + 1.5, input_y + 0.8, '畳み込み層\n(stride=1)', ha='center', fontsize=10, fontweight='bold')

# プーリング出力
pool_x = 9
pool_y = input_y + 0.5
# z1
circle1 = plt.Circle((pool_x, pool_y + 0.8), 0.35, fill=True, facecolor='lightyellow', edgecolor='black', linewidth=2)
ax.add_patch(circle1)
ax.text(pool_x, pool_y + 0.8, '$z_1$', ha='center', va='center', fontsize=11, fontweight='bold')

# z2
circle2 = plt.Circle((pool_x, pool_y - 0.8), 0.35, fill=True, facecolor='lightgreen', edgecolor='black', linewidth=2)
ax.add_patch(circle2)
ax.text(pool_x, pool_y - 0.8, '$z_2$', ha='center', va='center', fontsize=11, fontweight='bold')

ax.text(pool_x, input_y + 2.5, 'プーリング層\nMax Pooling', ha='center', fontsize=10, fontweight='bold')

# 全結合層
fc_x = 12
fc_nodes = ['$fc_0$', '$fc_1$', '$fc_2$']
fc_colors = ['coral', 'steelblue', 'purple']
for i, (label, color) in enumerate(zip(fc_nodes, fc_colors)):
    y = pool_y + 1.2 - i * 1.2
    circle = plt.Circle((fc_x, y), 0.35, fill=True, facecolor=color, edgecolor='black', linewidth=2, alpha=0.7)
    ax.add_patch(circle)
    ax.text(fc_x, y, label, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

ax.text(fc_x, input_y + 2.5, '全結合層', ha='center', fontsize=10, fontweight='bold')

# 出力層
out_x = 15
out_nodes = ['$P_0$', '$P_1$', '$P_2$']
for i, label in enumerate(out_nodes):
    y = pool_y + 1.2 - i * 1.2
    circle = plt.Circle((out_x, y), 0.35, fill=True, facecolor='lightgray', edgecolor='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(out_x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

ax.text(out_x, input_y + 2.5, '出力層\nSoftmax', ha='center', fontsize=10, fontweight='bold')

# 正解ラベル
target_x = 17
target_nodes = ['$t_0$', '$t_1$', '$t_2$']
for i, label in enumerate(target_nodes):
    y = pool_y + 1.2 - i * 1.2
    circle = plt.Circle((target_x, y), 0.3, fill=True, facecolor='lightgreen', edgecolor='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(target_x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

ax.text(target_x, input_y + 2.5, '正解値', ha='center', fontsize=10, fontweight='bold')

# 結合線（プーリング → 全結合）
for i in range(2):
    z_y = pool_y + 0.8 - i * 1.6
    for j in range(3):
        fc_y = pool_y + 1.2 - j * 1.2
        ax.plot([pool_x + 0.35, fc_x - 0.35], [z_y, fc_y], 'gray', alpha=0.5, linewidth=1)

# 重みラベルを追加
ax.text(10.5, pool_y + 1.5, '$w^{fc}_{1,0}$', fontsize=8, color='gray')
ax.text(10.5, pool_y + 0.3, '$w^{fc}_{1,1}$', fontsize=8, color='gray')
ax.text(10.5, pool_y - 0.9, '$w^{fc}_{1,2}$', fontsize=8, color='gray')

# 結合線（全結合 → 出力）
for i in range(3):
    for j in range(3):
        fc_y = pool_y + 1.2 - i * 1.2
        out_y = pool_y + 1.2 - j * 1.2
        ax.plot([fc_x + 0.35, out_x - 0.35], [fc_y, out_y], 'gray', alpha=0.3, linewidth=1)

# 結合線（出力 → 正解）
for i in range(3):
    y = pool_y + 1.2 - i * 1.2
    ax.annotate('', xy=(target_x - 0.3, y), xytext=(out_x + 0.35, y),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))

# 矢印
ax.annotate('', xy=(3.2, input_y), xytext=(2.3, input_y),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(5.7, input_y + 1.5), xytext=(4.5, input_y + 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(5.7, input_y - 1.5), xytext=(4.5, input_y - 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(8.5, pool_y + 0.8), xytext=(7.2, input_y + 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(8.5, pool_y - 0.8), xytext=(7.2, input_y - 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.set_xlim(-0.5, 18.5)
ax.set_ylim(1, 9)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('図3.20: 誤差逆伝播法を考察しやすくするための簡易的なCNN', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 各層の計算式

#### 畳み込み層の出力（式3-1相当）

$$c_{1,1}^1 = f(w_{1,1}^1 x_{1,1} + w_{1,2}^1 x_{1,2} + w_{2,1}^1 x_{2,1} + w_{2,2}^1 x_{2,2})$$

ここで $f$ はReLU関数です。簡略化のためバイアス項は省略しています。

#### プーリング層の出力

$$z_1 = \max(c_{1,1}^1, c_{1,2}^1, c_{2,1}^1, c_{2,2}^1)$$

#### 全結合層の出力（式3-4）

$$fc_0 = w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2$$
$$fc_1 = w_{1,1}^{fc} z_1 + w_{2,1}^{fc} z_2$$
$$fc_2 = w_{1,2}^{fc} z_1 + w_{2,2}^{fc} z_2$$

#### ソフトマックス関数による出力（式3-5）

$$P_0 = \frac{e^{fc_0}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

$$P_1 = \frac{e^{fc_1}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

$$P_2 = \frac{e^{fc_2}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

**重要**: $P_0 + P_1 + P_2 = 1$（確率の性質）

In [ ]:
# 簡略化CNNの計算例

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

# 仮の値を設定
z1, z2 = 0.8, 0.5  # プーリング層の出力

# 全結合層の重み（仮の値）
w_fc = np.array([
    [0.3, 0.5],   # fc0への重み (w_1,0, w_2,0)
    [0.7, 0.2],   # fc1への重み (w_1,1, w_2,1)
    [0.1, 0.4]    # fc2への重み (w_1,2, w_2,2)
])

z = np.array([z1, z2])

print("=== 簡略化CNNの計算例 ===\n")
print(f"プーリング層の出力: z1 = {z1}, z2 = {z2}\n")

# 全結合層の計算
fc = w_fc @ z
print("【全結合層の計算（式3-4）】")
print(f"fc0 = w_1,0 × z1 + w_2,0 × z2 = {w_fc[0,0]} × {z1} + {w_fc[0,1]} × {z2} = {fc[0]:.3f}")
print(f"fc1 = w_1,1 × z1 + w_2,1 × z2 = {w_fc[1,0]} × {z1} + {w_fc[1,1]} × {z2} = {fc[1]:.3f}")
print(f"fc2 = w_1,2 × z1 + w_2,2 × z2 = {w_fc[2,0]} × {z1} + {w_fc[2,1]} × {z2} = {fc[2]:.3f}")

# ソフトマックスの計算
print("\n【ソフトマックス関数（式3-5）】")
exp_fc = np.exp(fc)
sum_exp = np.sum(exp_fc)
print(f"e^fc0 = e^{fc[0]:.3f} = {exp_fc[0]:.4f}")
print(f"e^fc1 = e^{fc[1]:.3f} = {exp_fc[1]:.4f}")
print(f"e^fc2 = e^{fc[2]:.3f} = {exp_fc[2]:.4f}")
print(f"合計 = {sum_exp:.4f}")

P = softmax(fc)
print(f"\nP0 = {exp_fc[0]:.4f} / {sum_exp:.4f} = {P[0]:.4f}")
print(f"P1 = {exp_fc[1]:.4f} / {sum_exp:.4f} = {P[1]:.4f}")
print(f"P2 = {exp_fc[2]:.4f} / {sum_exp:.4f} = {P[2]:.4f}")
print(f"\n確率の合計: P0 + P1 + P2 = {np.sum(P):.4f} ✓")

## 6. 損失関数を定義する―対数尤度関数―（3-8節）

### 予測精度を数値化する

学習の目標は、正解クラスの予測確率を最大化することです。

例えば、正解が「1」のとき：
- $t_0 = 0, t_1 = 1, t_2 = 0$
- 理想的には $P_1$ が1に近づいてほしい

### 総乗記号 $\prod$（パイ）

**総乗記号**は、すべての項の積を計算する記号です：

$$\prod_{k=1}^{n} a_k = a_1 \times a_2 \times \cdots \times a_n$$

### 予測精度の数理モデル（式3-7）

正解クラスの予測値を表す数理モデル：

$$P = \prod_{k=0}^{2} P_k^{t_k}$$

**例: 正解が「1」のとき** ($t_0=0, t_1=1, t_2=0$)

$$P = P_0^{t_0} \times P_1^{t_1} \times P_2^{t_2} = P_0^0 \times P_1^1 \times P_2^0 = 1 \times P_1 \times 1 = P_1$$

→ 正解クラス $P_1$ の値だけが残る！

### 複数の入力画像への拡張（式3-8）

5枚の入力画像に対して：

$$P = \prod_{n=1}^{5} \prod_{k=0}^{2} P_{n,k}^{t_{n,k}}$$

- $n$: $n$枚目の入力画像
- $P_{n,k}$: $n$枚目の画像が数字$k$である確率
- $t_{n,k}$: $n$枚目の画像の正解が$k$なら1、そうでなければ0

In [ ]:
# 図3.22: 入力データ、確率値、正解の対応表

import pandas as pd

# 5枚の入力画像に対する予測確率と正解
data = {
    '入力No.': [1, 2, 3, 4, 5],
    'P0': [0.10, 0.75, 0.05, 0.05, 0.15],
    'P1': [0.80, 0.05, 0.85, 0.35, 0.65],
    'P2': [0.10, 0.20, 0.10, 0.60, 0.20],
    '正解': [1, 0, 1, 2, 1]
}

df = pd.DataFrame(data)

print("=== 図3.22: 入力データ、確率値、正解の対応表 ===\n")
print(df.to_string(index=False))

# 各入力に対して正解クラスの確率を強調表示
print("\n【正解クラスの予測確率】")
for i, row in df.iterrows():
    correct = int(row['正解'])
    p_correct = row[f'P{correct}']
    quality = "良い" if p_correct >= 0.8 else "改善の余地あり" if p_correct >= 0.6 else "低い"
    print(f"  入力{i+1}: 正解={correct}, P{correct}={p_correct:.2f} → {quality}")

In [ ]:
# 図3.22の可視化

fig, ax = plt.subplots(figsize=(14, 6))

# データ
inputs = [1, 2, 3, 4, 5]
P0 = [0.10, 0.75, 0.05, 0.05, 0.15]
P1 = [0.80, 0.05, 0.85, 0.35, 0.65]
P2 = [0.10, 0.20, 0.10, 0.60, 0.20]
correct = [1, 0, 1, 2, 1]

x = np.arange(len(inputs))
width = 0.25

bars1 = ax.bar(x - width, P0, width, label='P0', color='coral', edgecolor='black')
bars2 = ax.bar(x, P1, width, label='P1', color='steelblue', edgecolor='black')
bars3 = ax.bar(x + width, P2, width, label='P2', color='purple', edgecolor='black')

# 正解クラスのバーを強調
for i, c in enumerate(correct):
    if c == 0:
        bars1[i].set_edgecolor('gold')
        bars1[i].set_linewidth(3)
    elif c == 1:
        bars2[i].set_edgecolor('gold')
        bars2[i].set_linewidth(3)
    else:
        bars3[i].set_edgecolor('gold')
        bars3[i].set_linewidth(3)

ax.set_xlabel('入力画像', fontsize=12)
ax.set_ylabel('確率', fontsize=12)
ax.set_title('図3.22: 入力データ、確率値、正解の対応表\n（金色の枠 = 正解クラス）', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'入力{i}\n(正解:{c})' for i, c in zip(inputs, correct)])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 式(3-8)による予測精度Pの計算
print("\n=== 式(3-8)による予測精度Pの計算 ===\n")
P_product = 1.0
for i, (p0, p1, p2, c) in enumerate(zip(P0, P1, P2, correct)):
    probs = [p0, p1, p2]
    p_correct = probs[c]
    P_product *= p_correct
    print(f"入力{i+1}: 正解={c}, P{c}={p_correct:.2f}")

print(f"\nP = P1,1 × P2,0 × P3,1 × P4,2 × P5,1")
print(f"  = {P1[0]:.2f} × {P0[1]:.2f} × {P1[2]:.2f} × {P2[3]:.2f} × {P1[4]:.2f}")
print(f"  = {P_product:.6f}")
print(f"\n→ この値を1に近づけることが目標！")

### 対数尤度関数への変換（図3.21）

積の最大化は計算が複雑なので、**対数**を使って和の最小化に変換します。

#### 対数の性質

$$\log_a(mn) = \log_a m + \log_a n \quad \text{（積→和）}$$
$$\log_a(b^m) = m \log_a b \quad \text{（べき乗→係数）}$$

#### 損失関数の導出

$$L(P) = -\sum_{k=1}^{n} t_k \log P_k$$

この形式が**交差エントロピー損失**です。

- 積 $P = \prod P_k^{t_k}$ を最大化 ⟺ $-\log P = -\sum t_k \log P_k$ を最小化
- 正解クラスの確率が高いほど、損失は小さくなる

In [ ]:
# 対数尤度関数の計算

print("=== 対数尤度関数への変換 ===\n")

# 先ほどのデータを使用
P_correct = [P1[0], P0[1], P1[2], P2[3], P1[4]]  # 各入力の正解クラスの確率
correct_labels = [1, 0, 1, 2, 1]

print("【積の計算】")
P_product = np.prod(P_correct)
print(f"P = {' × '.join([f'{p:.2f}' for p in P_correct])}")
print(f"  = {P_product:.6f}")

print("\n【対数への変換】")
log_probs = [np.log(p) for p in P_correct]
print(f"log(P) = log({' × '.join([f'{p:.2f}' for p in P_correct])})")
print(f"       = {' + '.join([f'log({p:.2f})' for p in P_correct])}")
print(f"       = {' + '.join([f'{lp:.4f}' for lp in log_probs])}")
print(f"       = {np.sum(log_probs):.4f}")

print("\n【損失関数（負の対数）】")
loss = -np.sum(log_probs)
print(f"L = -log(P) = {loss:.4f}")

print("\n" + "=" * 50)
print("【重要ポイント】")
print(f"・積 P = {P_product:.6f} を最大化したい")
print(f"・これは損失 L = {loss:.4f} を最小化することと同じ")
print("・対数変換により、積→和になり計算が簡単に！")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 各入力の正解確率
ax = axes[0]
bars = ax.bar(range(1, 6), P_correct, color='steelblue', edgecolor='black')
ax.set_xlabel('入力画像', fontsize=12)
ax.set_ylabel('正解クラスの確率', fontsize=12)
ax.set_title('各入力の正解クラス確率', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 6))
ax.set_xticklabels([f'入力{i}\n(正解:{c})' for i, c in zip(range(1, 6), correct_labels)])
ax.set_ylim(0, 1)
for bar, p in zip(bars, P_correct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{p:.2f}', ha='center', fontsize=11)

# 右: -log(P)
ax = axes[1]
neg_log_probs = [-lp for lp in log_probs]
bars = ax.bar(range(1, 6), neg_log_probs, color='coral', edgecolor='black')
ax.set_xlabel('入力画像', fontsize=12)
ax.set_ylabel('-log(P)', fontsize=12)
ax.set_title('各入力の損失（-log P）\n確率が低いほど損失が大きい', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 6))
ax.set_xticklabels([f'入力{i}\n(P={p:.2f})' for i, p in zip(range(1, 6), P_correct)])
for bar, nlp in zip(bars, neg_log_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{nlp:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## まとめ

### CNNのパラメータ数（図3.18）

| 層 | 重み | バイアス |
|---|---|---|
| 畳み込み層1 | 50 | 2 |
| 畳み込み層2 | 72 | 4 |
| 全結合層 | 640 | 10 |
| **合計** | **762** | **16** |

**総パラメータ数: 778個**

### 簡略化CNNモデル（図3.20）

誤差逆伝播法を理解しやすくするための簡易モデル：
- 入力: 3×3ピクセル
- 出力: 3クラス（0, 1, 2）
- 各層の計算式（式3-4, 3-5）を明示

### 学習の仕組み（3-7節, 3-8節）

1. **予測精度の数理モデル（式3-7）**:
   $$P = \prod_{k=0}^{2} P_k^{t_k}$$
   正解クラスの確率だけが残る

2. **複数画像への拡張（式3-8）**:
   $$P = \prod_{n=1}^{5} \prod_{k=0}^{2} P_{n,k}^{t_{n,k}}$$

3. **対数尤度関数**:
   $$L(P) = -\sum_{k} t_k \log P_k$$
   積の最大化 → 和の最小化に変換

### 重要な数学的手法（図3.21）

| 数学的手法 | 情報工学的アプローチ |
|---|---|
| 対数の性質: $\log(mn) = \log m + \log n$ | 損失関数: $L(P) = -\sum t_k \log P_k$ |
| 総乗記号: $\prod_{k=1}^{n} a_k$ | 予測精度: $P = \prod P_k^{t_k}$ |

### 次のステップ

次回は**誤差逆伝播法**を使って、実際にパラメータの勾配を計算し、学習を行う方法を詳しく学びます。